In [42]:
import config
import importlib

importlib.reload(config)

import subprocess
import sys
import csv
import pandas as pd

from pathlib import Path


from config import (
    DATABASE_URL,
    SESSION_START,
    SESSIONS_SQL,
    SESSIONS_CSV,
    QUERIES,
    PARAMS
)
try:
    import psycopg2
except ImportError:
    print("Installing psycopg2 …")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "psycopg2"],
        check=True,
        timeout=120,
    )
    import psycopg2


In [ ]:
with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        for name, query in QUERIES.items():
            csv_path = query["csv"]

            if csv_path.exists():
                print(f"{name}: Datei bereits vorhanden - übersprungen.")
                continue

            cursor.execute(query["sql"], PARAMS)

            with csv_path.open(
                "w", newline="", encoding="utf-8"
            ) as file:
                writer = csv.writer(file)
                writer.writerow(column[0] for column in cursor.description)
                writer.writerows(cursor)

            print(f"{name}: gespeichert unter {csv_path}")

Sessions: Datei bereits vorhanden - übersprungen.
Trips: Datei bereits vorhanden - übersprungen.
Users: Datei bereits vorhanden - übersprungen.
Hotels: Datei bereits vorhanden - übersprungen.


In [44]:
# Load the raw data and show more rows/columns in tables
df = {
    name: pd.read_csv(query["csv"])
    for name, query in QUERIES.items()
}

df_sessions = df["Sessions"]
df_trips = df["Trips"]
df_users = df["Users"]
df_hotels = df["Hotels"]

display(df_sessions.head(), df_trips.head(), df_users.head(), df_hotels.head())

,session_id,user_id,trip_id,session_start,session_end,flight_booked,hotel_booked,page_clicks,gender,home_city,home_country,destination,base_fare_usd,hotel_name,nights
0,511998-eb4c8d214f36432bb8cdf6f8bf5fb145,511998,511998-a2cfa86a94fc4ca89867fe8cbac6d835,2023-07-06 03:32:00,2023-07-06 03:37:45,True,True,46,F,milwaukee,usa,denver,262.19,Crowne Plaza - denver,0.0
1,511998-c70655a5f5ab46f2a01dbb4dc1f29066,511998,NaN,2023-03-23 09:21:00,2023-03-23 09:21:15,False,False,2,F,milwaukee,usa,NaN,NaN,NaN,NaN
2,511998-12883b40234b47989a164ae8f8e7981d,511998,NaN,2023-03-19 11:48:00,2023-03-19 11:49:30,False,False,12,F,milwaukee,usa,NaN,NaN,NaN,NaN
3,511998-ef1066c461f3419eb31af9470c6d0bfe,511998,NaN,2023-02-09 17:06:00,2023-02-09 17:06:24,False,False,3,F,milwaukee,usa,NaN,NaN,NaN,NaN
4,511998-f2b9e8d07ee248589c8f880293f613e9,511998,NaN,2023-01-30 21:30:00,2023-01-30 21:30:37,False,False,5,F,milwaukee,usa,NaN,NaN,NaN,NaN


,trip_id,origin_airport,destination,base_fare_usd,hotel_name,nights,hotel_per_room_usd,user_id,session_id
0,511998-a2cfa86a94fc4ca89867fe8cbac6d835,MKE,denver,262.19,Crowne Plaza - denver,0.0,85.0,511998,511998-eb4c8d214f36432bb8cdf6f8bf5fb145
1,511998-9f63737fdd6b45f78af5478a54fd1841,MKE,portland,463.21,Crowne Plaza - portland,4.0,67.0,511998,511998-d85401653c4f4161ad692c1c1d40d65c
2,485505-2a013e36160d46d69645960c04213664,PHX,oklahoma city,238.87,Fairmont - oklahoma city,1.0,246.0,485505,485505-f14e6fc57abe48fb83e815d672204649
3,485505-dfc60f85f5d74256aeb4ee9cc12c46c9,PHX,san diego,85.75,Four Seasons - san diego,3.0,144.0,485505,485505-2cab94d048eb4a6db37cd68a5f4edf72
4,513569-44ddf06308a5427a8ca19ad56116a335,BOS,el paso,628.70,Accor - el paso,1.0,44.0,513569,513569-995b86eed1444a5fb3a9cc280886c43b


,user_id,birthdate,gender,married,has_children,home_country,home_city,home_airport,home_airport_lat,home_airport_lon,sign_up_date,total_sessions,total_trips_booked
0,23557,1958-12-08,F,True,False,usa,new york,LGA,40.777,-73.872,2021-07-22,12,4
1,94883,1972-03-16,F,True,False,usa,kansas city,MCI,39.297,-94.714,2022-02-07,12,3
2,101486,1972-12-07,F,True,True,usa,tacoma,TCM,47.138,-122.476,2022-02-17,13,4
3,101961,1980-09-14,F,True,False,usa,boston,BOS,42.364,-71.005,2022-02-17,12,7
4,106907,1978-11-17,F,True,True,usa,miami,TNT,25.862,-80.897,2022-02-24,14,3


,trip_id,hotel_name,nights,rooms,check_in_time,check_out_time,hotel_per_room_usd,gender,home_city,home_country,session_id,session_start
0,511998-a2cfa86a94fc4ca89867fe8cbac6d835,Crowne Plaza - denver,0,1,2023-07-13 14:47:49.515000,2023-07-14 11:00:00,85,F,milwaukee,usa,511998-eb4c8d214f36432bb8cdf6f8bf5fb145,2023-07-06 03:32:00
1,511998-9f63737fdd6b45f78af5478a54fd1841,Crowne Plaza - portland,4,1,2023-01-10 12:26:46.005000,2023-01-15 11:00:00,67,F,milwaukee,usa,511998-d85401653c4f4161ad692c1c1d40d65c,2023-01-05 18:31:00
2,485505-2a013e36160d46d69645960c04213664,Fairmont - oklahoma city,1,1,2023-07-27 13:40:21.810000,2023-07-29 11:00:00,246,F,phoenix,usa,485505-f14e6fc57abe48fb83e815d672204649,2023-07-18 17:19:00
3,485505-dfc60f85f5d74256aeb4ee9cc12c46c9,Four Seasons - san diego,3,2,2023-01-20 16:36:54,2023-01-24 11:00:00,144,F,phoenix,usa,485505-2cab94d048eb4a6db37cd68a5f4edf72,2023-01-14 15:03:00
4,513569-099c55cdb4204180a71de9522f22f604,Rosewood - winnipeg,2,1,2023-02-16 11:00:00,2023-02-18 11:00:00,51,F,boston,usa,513569-08d63afa1b5f499280ec620b3586c87e,2023-02-12 19:07:00


In [ ]:
pip install airports-py


Note: you may need to restart the kernel to use updated packages.


In [46]:
from airports import airport_data

for iata in df_users["home_airport"].head(10):
    airports = airport_data.get_airport_by_iata(iata)

    airport_name = airports[0]["airport"] if airports else "Nicht gefunden"
    print(f"{iata}: {airport_name}")

LGA: LaGuardia Airport
MCI: Kansas City International Airport
TCM: McChord Air Force Base
BOS: Logan International Airport
TNT: Dade Collier Training and Transition Airport
LAX: Los Angeles International Airport
SNA: John Wayne Airport
BIF: Biggs Army Air Field
AUS: Austin-Bergstrom International Airport
BHM: Birmingham-Shuttlesworth International Airport
